# Kadi-APY hands-on: from lab files to findable data

<a href="https://colab.research.google.com/github/msishuvo/thematic-workshop-kadiAPY/blob/main/notebooks/kadi_apy_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook you will use **kadi-apy** to put tensile test data into
**Kadi4Mat** automatically, and get it back out for analysis.

### How to use this notebook

| Do this | How |
|---|---|
| Run a cell | Click it and press **Shift + Enter** (or click ▶ on the left) |
| Run cells **in order**, top to bottom | Later cells need the results of earlier ones |
| Fill in a `TODO` | Change only the lines marked `TODO`, then run the cell |
| Stuck? | Click **💡 Show solution** below the exercise and copy it |
| Something is broken | *Runtime → Restart session*, then run all cells from the top |
| Need help | Write in the meeting chat or raise your hand 🙋 |

> **First:** *File → Save a copy in Drive*. Then your changes are saved in
> your own Google Drive.

---
## 0 · Setup (do this before the workshop)

### 0.1 Store your access token as a Colab secret 🔑

A **personal access token (PAT)** is like a password for programs. We keep it
out of the notebook so you can share your notebook safely.

1. Click the **🔑 key icon** in the left sidebar ("Secrets").
2. Click **Add new secret**. Name: `KADI_PAT`, Value: your personal access token.
3. Switch on **Notebook access**.

### 0.2 Install kadi-apy and download the workshop data

Run the cell below. It takes about 30 seconds.

In [ ]:
%pip install -q kadi-apy

import os, subprocess, sys

REPO_DIR = "/content/thematic-workshop-kadiAPY"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/msishuvo/thematic-workshop-kadiAPY.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.append(os.path.join(REPO_DIR, "exercises"))

print("✅ Ready. Data files:", sorted(os.listdir("data/tensile_tests")))

### 0.3 Connect to Kadi4Mat

This writes the Kadi4Mat address and your token into the kadi-apy configuration file and
checks that the connection works. You should see **✅ Connected**.

In [ ]:
import configparser
from pathlib import Path
from google.colab import userdata

host = "https://kadi4mat.iam.kit.edu"

try:
    pat = userdata.get("KADI_PAT").strip()
except Exception as error:
    raise RuntimeError(
        "❌ Could not read the secret KADI_PAT. Check step 0.1: is the name "
        "spelled exactly like this, and is 'Notebook access' switched on?"
    ) from error

config = configparser.ConfigParser()
config["global"] = {"default": "workshop"}
config["workshop"] = {"host": host, "pat": pat}
config_path = Path.home() / ".kadiconfig"
with open(config_path, "w") as f:
    config.write(f)
config_path.chmod(0o600)

from kadi_apy import KadiManager

manager = KadiManager()
me = manager.pat_user.meta
print(f"✅ Connected to {host} as {me.get('displayname', me.get('id'))}")

### 0.4 Choose your personal prefix

Everything you create gets this prefix, so your records don't clash with
other participants'. Use your initials, **lowercase letters only**,
e.g. `jd` for Jane Doe.

In [ ]:
PREFIX = "xy"  # TODO: change to your initials

import re
assert re.fullmatch(r"[a-z0-9]{2,10}", PREFIX), "Use 2-10 lowercase letters/digits only."
assert PREFIX != "xy", "Please change 'xy' to your own initials 🙂"
CAMPAIGN_TAG = f"{PREFIX}-tensile-campaign"
print("✅ Your prefix is", PREFIX)

---
## 1 · The command line: no Python needed (15 min)

Everything you click in the Kadi4Mat web interface can also be done with
**commands**. Lines starting with `!` are terminal commands; `{PREFIX}` is
replaced with your prefix automatically.

Keep Kadi4Mat open in another browser tab and watch what happens after each cell.

**1.1 Create a record** for sample S01:

In [ ]:
!kadi-apy records create -i {PREFIX}-sample-s01 -t "Steel sample S01"

**1.2 Upload the measurement file.** `-R` selects the record by its identifier, `-n` is the file:

In [ ]:
!kadi-apy records add-files -R {PREFIX}-sample-s01 -n data/tensile_tests/S01_tensile.csv

**1.3 Add metadata.** Each entry has a name (`-m`), value (`-v`), type (`-t`) and optionally a unit (`-u`):

In [ ]:
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m material -v "S355 steel"
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m specimen_length -v 80 -t float -u mm
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m test_temperature -v 23 -t float -u degC

**1.4 Add a tag**, then **1.5 look at the result** (`-m` metadata, `-l` file list):

In [ ]:
!kadi-apy records add-tag -R {PREFIX}-sample-s01 -t {CAMPAIGN_TAG}
!kadi-apy records show-info -R {PREFIX}-sample-s01 -m -l

✋ **Try it yourself:** add a metadatum `operator` with your name to the record.
Copy one of the lines from 1.3 into the cell below and change it.

In [ ]:
# TODO: add a metadatum "operator" with your name

<details><summary><b>💡 Show solution</b> (click)</summary>

```python
!kadi-apy records add-metadatum -R {PREFIX}-sample-s01 -m operator -v "Jane Doe"
```

</details>

---
## 2 · Your first Python script (20 min)

The same steps in Python. Python is better when you need logic: loops,
calculations, reading files.

Three ideas you need:
- `manager` is "your login" (created in step 0.3).
- `manager.record(identifier=..., title=..., create=True)` means
  **get the record, or create it if it doesn't exist.** Running a cell twice does not create duplicates.
- Metadata is a **list of small dictionaries**: `key`, `type`, `value` and optionally `unit`.

In [ ]:
record = manager.record(
    identifier=f"{PREFIX}-sample-s02",
    title="Steel sample S02",
    create=True,
)
print("Working on:", record)

metadata = [
    {"key": "material", "type": "str", "value": "S355 steel"},
    {"key": "specimen_length", "type": "float", "value": 80.0, "unit": "mm"},
    # TODO 1: add "test_temperature" = 23.0 degC (copy the line above and change it)
]
record.add_metadata(metadata, force=True)

# TODO 2: upload the file "data/tensile_tests/S02_tensile.csv"
#         Hint: record.upload_file("path/to/file", force=True)

record.add_tag(CAMPAIGN_TAG)
print("✅ Done! Look at the record in Kadi4Mat.")

<details><summary><b>💡 Show solution</b> (click)</summary>

```python
record = manager.record(
    identifier=f"{PREFIX}-sample-s02",
    title="Steel sample S02",
    create=True,
)
print("Working on:", record)

metadata = [
    {"key": "material", "type": "str", "value": "S355 steel"},
    {"key": "specimen_length", "type": "float", "value": 80.0, "unit": "mm"},
    {"key": "test_temperature", "type": "float", "value": 23.0, "unit": "degC"},
]
record.add_metadata(metadata, force=True)

record.upload_file("data/tensile_tests/S02_tensile.csv", force=True)

record.add_tag(CAMPAIGN_TAG)
print("✅ Done! Look at the record in Kadi4Mat.")
```

</details>

---
## 3 · Batch upload a whole test campaign (20 min)

**The everyday lab situation:** the testing machine produced a folder of result
files. Instead of uploading them one by one and copying numbers into metadata
fields by hand, we let Python do it for **every file in the folder**.

First, look at what the helper function `evaluate` gets out of one file.
(In your own lab, this function would read *your* instrument's files.)

In [ ]:
from tensile import evaluate, read_curve

evaluate("data/tensile_tests/S01_tensile.csv")

Now the batch upload. A `for` loop means: **do the indented block once for every file.**
Fill in the three `TODO`s.

In [ ]:
from pathlib import Path

UNITS = {"youngs_modulus": "GPa", "yield_strength": "MPa",
         "tensile_strength": "MPa", "elongation_at_break": "%"}

# One collection for the whole campaign
collection = manager.collection(
    identifier=f"{PREFIX}-tensile-campaign-2026",
    title=f"Tensile test campaign 2026 ({PREFIX})",
    create=True,
)

for csv_file in sorted(Path("data/tensile_tests").glob("*.csv")):
    sample_id = csv_file.stem.split("_")[0]  # "S01_tensile" -> "S01"

    record = manager.record(
        identifier=f"{PREFIX}-sample-{sample_id.lower()}",
        title=f"Steel sample {sample_id}",
        create=True,
    )

    results = evaluate(csv_file)
    metadata = [
        {"key": "material", "type": "str", "value": "S355 steel"},
        {"key": "test_standard", "type": "str", "value": "ISO 6892-1"},
    ]
    for key, value in results.items():
        metadata.append({"key": key, "type": "float", "value": value, "unit": UNITS[key]})

    # TODO 1: add the metadata to the record (use force=True)

    # TODO 2: upload the CSV file (upload_file needs text: str(csv_file))

    # TODO 3: add the tag CAMPAIGN_TAG

    record.add_collection_link(collection.id)
    print(f"✅ {sample_id}: {results}")

print("Done! Open the collection in Kadi4Mat.")

<details><summary><b>💡 Show solution</b> (click)</summary>

```python
    record.add_metadata(metadata, force=True)
    record.upload_file(str(csv_file), force=True)
    record.add_tag(CAMPAIGN_TAG)
```

</details>

🚀 **Faster than the others?** Create a record for the testing machine and link every
sample to it:
```python
machine = manager.record(identifier=f"{PREFIX}-tensile-machine",
                         title="Tensile testing machine", create=True)
record.link_record(machine.id, name="tested with")
```

---
## 4 · Find, download and plot (10 min)

Your data is in Kadi4Mat. Now get it back: search by tag, print a summary
table, download the files and plot all curves together.

In [ ]:
import matplotlib.pyplot as plt

found = manager.search.search_resources("record", tag=[CAMPAIGN_TAG], per_page=100).json()["items"]
print(f"Found {len(found)} records with tag '{CAMPAIGN_TAG}'\n")

os.makedirs("downloads", exist_ok=True)
print(f"{'Record':<22} {'Yield [MPa]':>12} {'UTS [MPa]':>10}")
for item in found:
    record = manager.record(id=item["id"])
    extras = {e["key"]: e.get("value") for e in record.meta["extras"]}
    print(f"{record.meta['title']:<22} {extras.get('yield_strength', '-'):>12} "
          f"{extras.get('tensile_strength', '-'):>10}")

    for file_info in record.get_filelist().json()["items"]:
        if file_info["name"].endswith(".csv"):
            local_path = f"downloads/{file_info['name']}"
            record.download_file(file_info["id"], local_path)
            strain, stress = read_curve(local_path)
            plt.plot(strain, stress, label=record.meta["title"])

# TODO: label the axes "Strain [%]" and "Stress [MPa]"  (plt.xlabel(...), plt.ylabel(...))

plt.title("Tensile test campaign")
plt.legend()
plt.show()

<details><summary><b>💡 Show solution</b> (click)</summary>

```python
plt.xlabel("Strain [%]")
plt.ylabel("Stress [MPa]")
```

</details>

---
## 🎉 Well done! What next?

**Try it with your own data:** upload a few of your own files with the 📁 folder icon
in the left sidebar, change the folder name in section 3, and adapt `evaluate`
(or just skip it and upload files with basic metadata).

> ⚠️ Colab runs on Google servers. Before using real (unpublished) lab data here,
> check your institution's data policy. kadi-apy also runs on your own computer:
> `pip install kadi-apy`.

**Useful links**
- kadi-apy documentation: https://kadi-apy.readthedocs.io/en/stable/
- kadi-apy source code and issues: https://gitlab.com/iam-cms/kadi-apy
- Workshop material: https://github.com/msishuvo/thematic-workshop-kadiAPY